# 과수 질병 데이터 로컬 JSON 클래스 매핑

이 노트북은 `data/labels`의 JSON에서 `crop`과 `disease` 조합을 집계하고, 사용자가 승인한 로컬 코드 매핑을 9개 고유 학습 클래스 ID에 연결한 과정을 설명합니다. 원본 이미지나 JSON은 수정하지 않습니다.

**실행 방법:** 저장소 루트(`C:\\Users\\mbc\\aws`)를 현재 폴더로 두고 셀을 위에서부터 실행하세요. 필요한 입력은 `config/local_disease_mapping.csv`, `config/official_disease_codebook.csv`, `data/labels`입니다.

> 매핑의 질병명은 사용자가 제시한 클래스 순서와 로컬 JSON 코드에 따라 지정한 것입니다. 공식 전사표와 사과 질병 코드가 다르므로, 원본 AI Hub 코드북과의 일치를 의미하지 않습니다.

## 1. 매핑 규칙

원본 `disease` 값은 작물별로 해석해야 합니다. 예를 들어 `disease=0`은 배 정상과 사과 정상 모두에 있으므로, 원본 숫자를 그대로 YOLO ID로 사용하면 두 클래스를 구별할 수 없습니다. `(crop, disease)` 조합을 최종 ID로 변환합니다.

| YOLO ID | 클래스명 | crop | 로컬 disease |
|---:|---|---:|---:|
| 0 | `pear_healthy` | 1 | 0 |
| 1 | `pear_scab` | 1 | 1 |
| 2 | `pear_fire_blight` | 1 | 2 |
| 3 | `apple_healthy` | 2 | 0 |
| 4 | `apple_brown_spot` | 2 | 3 |
| 5 | `apple_fire_blight` | 2 | 4 |
| 6 | `apple_valsa_canker` | 2 | 5 |
| 7 | `apple_alternaria_leaf_spot` | 2 | 6 |
| 8 | `apple_anthracnose` | 2 | 7 |

In [ ]:
from collections import Counter
import csv
import json
from pathlib import Path

# 저장소 루트가 현재 경로가 아니어도, 상위 경로에서 설정 폴더를 찾습니다.
candidates = (Path.cwd(), *Path.cwd().parents)
ROOT = next((path for path in candidates if (path / "config" / "dataset_mapping.yaml").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("저장소 루트(config/dataset_mapping.yaml)를 찾지 못했습니다.")

LABEL_ROOT = ROOT / "data" / "labels"
LOCAL_MAPPING_PATH = ROOT / "config" / "local_disease_mapping.csv"
OFFICIAL_MAPPING_PATH = ROOT / "config" / "official_disease_codebook.csv"
assert LABEL_ROOT.is_dir(), f"라벨 폴더가 없습니다: {LABEL_ROOT}"
assert LOCAL_MAPPING_PATH.is_file(), f"로컬 매핑 CSV가 없습니다: {LOCAL_MAPPING_PATH}"

print(f"저장소: {ROOT}")
print(f"라벨 폴더: {LABEL_ROOT}")
print(f"로컬 매핑표: {LOCAL_MAPPING_PATH}")

## 2. 로컬 JSON 분포 집계

모든 하위 폴더의 JSON을 읽어 `crop+disease` 조합과 라벨 폴더별 조합을 셉니다. JSON 파싱 실패나 필드 누락은 무시하지 않고 오류 목록에 남깁니다.

In [ ]:
pair_counts = Counter()
folder_pair_counts = Counter()
json_errors = []
json_paths = sorted(LABEL_ROOT.rglob("*.json"))

for json_path in json_paths:
    try:
        with json_path.open(encoding="utf-8-sig") as stream:
            payload = json.load(stream)
        annotations = payload.get("annotations") or {}
        crop = annotations.get("crop")
        disease = annotations.get("disease")
        if crop is None or disease is None:
            raise ValueError("annotations.crop 또는 annotations.disease가 null/누락")
        pair = (int(crop), int(disease))
        pair_counts[pair] += 1
        folder_pair_counts[(json_path.parent.name, *pair)] += 1
    except (OSError, UnicodeError, json.JSONDecodeError, TypeError, ValueError) as exc:
        json_errors.append((str(json_path), str(exc)))

print(f"JSON 파일 수: {len(json_paths):,}")
print(f"정상 집계 조합 수: {sum(pair_counts.values()):,}")
print(f"파싱/필드 오류: {len(json_errors):,}")
print("\n로컬 crop-disease 조합 빈도")
for (crop, disease), count in sorted(pair_counts.items()):
    print(f"crop={crop}, disease={disease}: {count:,}")

if json_errors:
    print("\n오류 예시(최대 10개)")
    for path, reason in json_errors[:10]:
        print(f"- {path}: {reason}")

## 3. 승인된 로컬 매핑과 관측값 대조

CSV의 각 행은 `(crop, disease)`를 고유한 클래스 ID/이름과 연결합니다. 설정에 없는 관측 조합이나 데이터에서 빠진 매핑 조합이 있으면 아래 검증에서 즉시 중단합니다.

In [ ]:
with LOCAL_MAPPING_PATH.open(encoding="utf-8-sig", newline="") as stream:
    mapping_rows = list(csv.DictReader(stream))

mapping_by_pair = {
    (int(row["crop"]), int(row["disease"])): {
        "class_id": int(row["class_id"]),
        "class_name": row["class_name"],
    }
    for row in mapping_rows
}
assert len(mapping_by_pair) == len(mapping_rows), "매핑 CSV에 crop+disease 중복이 있습니다."
assert sorted(item["class_id"] for item in mapping_by_pair.values()) == list(range(9)), "클래스 ID는 0~8이어야 합니다."
assert len({item["class_name"] for item in mapping_by_pair.values()}) == 9, "클래스 이름이 중복됩니다."

observed_pairs = set(pair_counts)
configured_pairs = set(mapping_by_pair)
unknown_pairs = observed_pairs - configured_pairs
missing_pairs = configured_pairs - observed_pairs
assert not json_errors, f"JSON 오류 {len(json_errors)}건이 있어 매핑을 검증할 수 없습니다."
assert not unknown_pairs, f"매핑 설정에 없는 로컬 조합: {sorted(unknown_pairs)}"
assert not missing_pairs, f"로컬 데이터에 없는 매핑 조합: {sorted(missing_pairs)}"

print("매핑 검증: 통과")
print("YOLO ID | 클래스명 | crop | disease | 이미지 수")
for pair, item in sorted(mapping_by_pair.items(), key=lambda entry: entry[1]["class_id"]):
    print(f"{item['class_id']:>7} | {item['class_name']:<30} | {pair[0]:>4} | {pair[1]:>7} | {pair_counts[pair]:>8}")

## 4. 공식표와의 차이

공식표 파일은 이전에 사용자가 제공한 표를 옮긴 참고 자료입니다. 로컬 JSON과 다르게 기록된 코드를 조용히 이동시키거나 덮어쓰지 않습니다. 여기서는 양쪽의 조합을 나란히 비교해 차이를 명시합니다.

In [ ]:
if OFFICIAL_MAPPING_PATH.is_file():
    with OFFICIAL_MAPPING_PATH.open(encoding="utf-8-sig", newline="") as stream:
        official_rows = list(csv.DictReader(stream))
    official_pairs = {(int(row["crop"]), int(row["disease"])) for row in official_rows}
    print(f"공식표 전사 행 수: {len(official_rows)}")
    print(f"공식표에만 있는 조합: {sorted(official_pairs - observed_pairs)}")
    print(f"로컬 JSON에만 있는 조합: {sorted(observed_pairs - official_pairs)}")
    print("\n공식표와 로컬 관측 수 비교")
    for row in official_rows:
        pair = (int(row["crop"]), int(row["disease"]))
        print(f"{pair}: official={row['official_disease_name']}, local_count={pair_counts[pair]}")
else:
    print("공식표 파일이 없습니다. 로컬 JSON과의 비교를 생략합니다.")

### 해석 및 주의사항

- 기존 표에는 배 정상과 사과 정상의 클래스 ID가 둘 다 `0`으로 적혀 있어, 해당 열은 작물별 원본 질병 코드로 해석했습니다. 학습 클래스 ID는 9개를 구분하도록 `0~8`로 새로 부여했습니다.
- 현재 로컬 JSON은 사과 정상 `(2,0)` 100개, 사과 질병 코드 `(2,3)`~`(2,7)` 각 100개입니다. 제공된 기존 공식표는 사과 정상 `(2,3)`, 질병 `(2,4)`~`(2,8)`로 기록돼 있어 일치하지 않습니다.
- 아래 매핑은 사용자가 승인한 **로컬 JSON 기준 분류**입니다. 질병명-코드 관계가 AI Hub 원본 코드북으로 독립 검증된 것은 아닙니다.

## 5. 생성된 Classification 결과 확인

변환 결과의 `metadata.csv`에서 클래스별 실제 이미지 수, split별 비율을 다시 집계합니다. 이미지 원본/JSON은 그대로 두고 별도의 `data/processed/classification`에 복사했습니다.

In [ ]:
OUTPUT_ROOT = ROOT / "data" / "processed" / "classification"
metadata_path = OUTPUT_ROOT / "metadata.csv"
assert metadata_path.is_file(), f"변환 결과를 찾지 못했습니다: {metadata_path}"

with metadata_path.open(encoding="utf-8-sig", newline="") as stream:
    metadata_rows = list(csv.DictReader(stream))

counts_by_split_class = Counter((row["split"], row["class_name"]) for row in metadata_rows)
assert len(metadata_rows) == 1200, f"예상 1,200개와 다릅니다: {len(metadata_rows)}"
assert {row["class_id"] for row in metadata_rows} == {str(value) for value in range(9)}
print(f"출력 경로: {OUTPUT_ROOT}")
print(f"총 이미지: {len(metadata_rows):,}")
for split in ("train", "val", "test"):
    split_count = sum(count for (name, _), count in counts_by_split_class.items() if name == split)
    print(f"{split}: {split_count:,} ({split_count / len(metadata_rows):.1%})")

print("\nsplit별 클래스 수")
for split in ("train", "val", "test"):
    print(f"\n[{split}]")
    for item in sorted(mapping_by_pair.values(), key=lambda value: value["class_id"]):
        count = counts_by_split_class[(split, item["class_name"])]
        print(f"ID {item['class_id']}: {item['class_name']} = {count}")

## 작업 결과 요약

- JSON 1,200개를 검사했고, crop-disease 조합 9종이 모두 매핑표와 일치했습니다.
- 9개 학습 클래스에 이미지가 모두 배정됐습니다: 배 정상 100, 배 검은별무늬병 250, 배 과수화상병 250, 사과 정상 100, 사과 질병 5종 각 100.
- Full-scene YOLO Classification 데이터셋 1,200장을 생성했습니다. 분할은 train 797, val 264, test 139입니다. 그룹 경계 때문에 목표 70/20/10과 정확히 일치하지 않습니다.
- 파일명/해시/유사도 기반 그룹 분할은 승인받았으나 실제 동일 개체·촬영 세션을 보증하지는 않으므로 잔여 누수 위험이 있습니다.
- 입력 포맷의 `points`가 주목 객체 bbox라는 사용자 정의에 따라 YOLO Detection 이미지 1,200장, TXT 라벨 1,200개, `data.yaml`을 생성했습니다. 박스는 병변이 아닌 주목 객체 위치입니다.
- Classification/Detection Colab 및 Google Drive 업로드용 ZIP을 각각 생성했습니다. Google Drive 계정에 직접 업로드한 것은 아니며, 아래 셀에서 ZIP 구조와 라벨을 검증할 수 있습니다.
- `area` 값 8~11과 `grow` 값 21~26은 제공 표에 적힌 범위와 다릅니다. 값을 임의 변경하지 않았고 보조 메타데이터로 보존했습니다.
- 상세 매핑: `reports/dataset/class_mapping_report.md`; 포맷 검사: `reports/dataset/annotation_format_review.md`; split 상세: `reports/dataset/data_split_report.csv`; 결과 요약: `reports/dataset/conversion_summary.md`.

## 6. 어노테이션 포맷과 Detection 변환

JSON 포맷에서 `annotations.points`는 주목 객체의 `{xtl, ytl, xbr, ybr}` bbox입니다. 이 박스를 병변 위치라고 해석하지 않습니다. 전체 데이터 1,200건에서 JSON 파싱/필수 구조 오류, 이미지 크기 불일치, bbox 좌표 범위 오류는 0건이었습니다.

YOLO Detection은 `class_id x_center y_center width height` 형식을 사용합니다. 좌표는 실제 이미지 너비/높이로 나눠 `[0,1]` 정규화하고, 유효하지 않은 박스는 자르거나 수정하지 않고 오류로 처리합니다. `area`와 `grow`는 보조 정보로 metadata에 남기며 클래스 ID 계산에는 쓰지 않습니다.

In [ ]:
import csv
import json
from zipfile import ZipFile

import yaml

PROCESSED_ROOT = ROOT / "data" / "processed"
DETECTION_ROOT = PROCESSED_ROOT / "detection"
DETECTION_ZIP = PROCESSED_ROOT / "detection.zip"
CLASSIFICATION_ZIP = PROCESSED_ROOT / "classification.zip"

# Detection data.yaml의 클래스 순서와 이미지/라벨 경로를 확인합니다.
with (DETECTION_ROOT / "data.yaml").open(encoding="utf-8") as stream:
    detection_config = yaml.safe_load(stream)
assert detection_config["nc"] == 9
assert detection_config["names"] == [item["class_name"] for item in sorted(mapping_by_pair.values(), key=lambda value: value["class_id"])]
assert all(detection_config[split] == f"images/{split}" for split in ("train", "val", "test"))

detection_metadata_path = DETECTION_ROOT / "metadata.csv"
with detection_metadata_path.open(encoding="utf-8-sig", newline="") as stream:
    detection_metadata = list(csv.DictReader(stream))
assert len(detection_metadata) == 1200
assert {row["annotation_semantics"] for row in detection_metadata} == {"approved_subject_object_not_lesion"}

# 각 이미지에 대응하는 라벨 파일이 있고, 모든 정규화 좌표가 유효한지 검사합니다.
label_paths = sorted((DETECTION_ROOT / "labels").rglob("*.txt"))
image_paths = [path for split in ("train", "val", "test") for path in (DETECTION_ROOT / "images" / split).iterdir() if path.is_file()]
assert len(image_paths) == len(label_paths) == 1200
box_count = 0
for label_path in label_paths:
    for line in label_path.read_text(encoding="ascii").splitlines():
        class_id, *coordinates = line.split()
        assert 0 <= int(class_id) < 9
        assert len(coordinates) == 4
        assert all(0.0 <= float(value) <= 1.0 for value in coordinates)
        box_count += 1
assert box_count == 1200

# ZIP 파일이 열리고, 필수 파일과 전체 이미지/라벨이 들어 있는지 확인합니다.
with ZipFile(DETECTION_ZIP) as archive:
    members = archive.namelist()
    assert "data.yaml" in members and "metadata.csv" in members
    assert sum(name.startswith("images/") and name.lower().endswith((".jpg", ".jpeg", ".png", ".webp")) for name in members) == 1200
    assert sum(name.startswith("labels/") and name.endswith(".txt") for name in members) == 1200
with ZipFile(CLASSIFICATION_ZIP) as archive:
    members = archive.namelist()
    assert "classes.json" in members and "metadata.csv" in members
    assert sum(name.lower().endswith((".jpg", ".jpeg", ".png", ".webp")) for name in members) == 1200

print(f"Detection: {len(image_paths):,} images, {len(label_paths):,} TXT files, {box_count:,} boxes")
print(f"Classification ZIP: {CLASSIFICATION_ZIP.stat().st_size / (1024**3):.2f} GiB")
print(f"Detection ZIP: {DETECTION_ZIP.stat().st_size / (1024**3):.2f} GiB")
print("Detection data.yaml, normalized labels, ZIP contents, and object-not-lesion semantics: verified")

## 7. Google Drive 업로드 및 Colab 사용

ZIP은 로컬 `data/processed`에 생성되어 있습니다. 이 환경에서는 Google Drive에 직접 업로드하지 않았습니다. 아래 순서로 두 ZIP 중 원하는 학습 작업의 파일을 Drive에 업로드한 뒤 Colab에서 사용하세요.

1. 로컬 파일 탐색기에서 `data/processed/classification.zip` 또는 `data/processed/detection.zip`을 Google Drive 폴더에 업로드합니다. 각각 약 1.89 GiB이므로 Drive 용량과 업로드 연결을 확인하세요.
2. Colab에서 Drive를 마운트하고, 업로드한 ZIP 경로를 지정해 `/content`에 압축을 풉니다.
3. Classification은 `model.train(data='/content/orchard_classification', ...)`를, Detection은 `model.train(data='/content/orchard_detection/data.yaml', ...)`를 사용합니다.

ZIP 전체를 Colab에 업로드하는 방식보다 Drive 경로에서 직접 압축을 풀면 Colab 브라우저 업로드 제한의 영향을 줄일 수 있습니다.

In [ ]:
# Google Colab에서 실행할 예시입니다. Drive의 파일 경로를 본인 폴더에 맞게 수정하세요.
from google.colab import drive
from pathlib import Path
from zipfile import ZipFile

drive.mount('/content/drive')
classification_zip = Path('/content/drive/MyDrive/datasets/classification.zip')
detection_zip = Path('/content/drive/MyDrive/datasets/detection.zip')

# 사용할 작업 한쪽만 압축 해제해도 됩니다.
with ZipFile(classification_zip) as archive:
    archive.extractall('/content/orchard_classification')

# Detection을 학습할 때 실행합니다.
# with ZipFile(detection_zip) as archive:
#     archive.extractall('/content/orchard_detection')

# Classification 예시:
# from ultralytics import YOLO
# model = YOLO('yolo11n-cls.pt')
# model.train(data='/content/orchard_classification', epochs=50, imgsz=224)

# Detection 예시. 출력은 병변 위치가 아니라 주목 객체 위치입니다.
# model = YOLO('yolo11n.pt')
# model.train(data='/content/orchard_detection/data.yaml', epochs=50, imgsz=640)